# 01 — Nettoyage, partie 1 : comparer les formats (ticket SM-004)

**Demande de Mame Diarra :** quel format prendre pour quoi, avec des chiffres.

**Protocole.** Table `lignes_commande` de la partition volume. Quatre formats. Temps = médiane de
5 lectures après une lecture de chauffe ignorée : la mesure est donc **à chaud** (fichiers en cache RAM).
Taille sur disque. Types inférés par Pandas avec ses réglages par défaut.

In [1]:
import os
from pathlib import Path

import pandas as pd

# Le notebook vit dans notebooks/ : on remonte à la racine du dépôt pour que les chemins
# relatifs (data/...) fonctionnent comme en ligne de commande.
if Path.cwd().name == "notebooks":
    os.chdir(Path.cwd().parent)

from senmarket.formats import LECTEURS, comparer, en_markdown, lire_par_morceaux

RUN_DATE = "2026-11-06"          # partition volume (200 000 lignes de commande)
SRC = Path(f"data/raw/date={RUN_DATE}/lignes_commande.csv")
SORTIE = Path("data/formats")    # ignoré par Git
print(SRC, SRC.exists())

data/raw/date=2026-11-06/lignes_commande.csv True


## 1. Exports, temps de lecture, taille

In [2]:
mesures = comparer(SRC, SORTIE, dates=[], colonne="quantite")
print(en_markdown(mesures, "quantite"))

| format | temps médian (s) | taille (Mo) | type de `quantite` |
|---|---|---|---|
| csv | 0.054 | 9.2 | int64 |
| json | 0.918 | 29.7 | int64 |
| jsonl | 1.029 | 29.7 | int64 |
| parquet | 0.033 | 2.8 | int64 |


## 2. Types inférés par Pandas, colonne par colonne

In [3]:
types = pd.DataFrame({f: LECTEURS[f](SORTIE / f"lignes_commande.{f}").dtypes.astype(str)
                      for f in LECTEURS})
types

,csv,json,jsonl,parquet
ligne_commande_id,str,str,str,str
commande_id,str,str,str,str
produit_id,str,str,str,str
quantite,int64,int64,int64,int64
prix_unitaire_xof,int64,int64,int64,int64
montant_xof,str,str,str,str


`montant_xof` est de type `str` dans les 4 formats. **Hypothèse à vérifier ci-dessous** : la colonne
contient du texte (défaut `montant_texte`) en plus de valeurs vides (défaut `valeurs_manquantes`).

In [4]:
brut = pd.read_csv(SRC, dtype=str)
vides = brut["montant_xof"].isna()
illisibles = pd.to_numeric(brut["montant_xof"], errors="coerce").isna() & ~vides
print("lignes :", len(brut), "| montants vides :", int(vides.sum()),
      "| montants texte :", int(illisibles.sum()))
brut.loc[illisibles, "montant_xof"].head()

lignes : 210000 | montants vides : 16000 | montants texte : 6000


18      12080 F CFA
64       6000 F CFA
110      6 000 FCFA
190    450000 F CFA
210       3.500 XOF
Name: montant_xof, dtype: str

**Résultat.** 16 000 montants vides et 6 000 montants en texte (« 12080 F CFA »,
« 6 000 FCFA », « 3.500 XOF »...) : c'est ce qui force le type `str`. Piège pour le chapitre 4 :
« 3.500 XOF » vaut 3 500, pas 3,5. Le CSV compte 210 000 lignes pour 200 000 générées : écart à
rapprocher de `bilan_generation.csv` (défaut `doublon_ligne`).

## 3. Lecture par morceaux (`chunksize`) : même résultat qu'une lecture complète ?

In [5]:
complet = pd.read_csv(SRC)
morceaux = lire_par_morceaux(SRC, lignes_par_morceau=50_000)
pd.testing.assert_frame_equal(complet, morceaux)   # lève une erreur s'ils diffèrent
print("identique, forme :", complet.shape)

identique, forme : (210000, 6)


## 4. Le type de `date_commande` (question 1)
`lignes_commande` n'a pas de date : on mesure donc la table `commandes`, lue avec `parse_dates`
avant l'export.

In [6]:
SRC_CMD = Path(f"data/raw/date={RUN_DATE}/commandes.csv")
m_cmd = comparer(SRC_CMD, SORTIE, dates=["date_commande"], colonne="date_commande")
print(en_markdown(m_cmd, "date_commande"))

| format | temps médian (s) | taille (Mo) | type de `date_commande` |
|---|---|---|---|
| csv | 0.021 | 3.1 | str |
| json | 0.445 | 8.7 | str |
| jsonl | 0.387 | 8.7 | str |
| parquet | 0.022 | 0.9 | datetime64[us] |


## 5. Conclusion par usage (brouillon de Claude : à reformuler avec tes mots)

| Usage | Format retenu | Argument chiffré | Alternative écartée |
|---|---|---|---|
| Lire à l'œil | CSV | s'ouvre partout ; 9,2 Mo contre 29,7 Mo en JSON | JSON : 3,2 fois plus lourd |
| Ingérer en flux | JSON Lines | un enregistrement par ligne, ajout en fin de fichier (propriété du format, non mesurée) ; lecture complète ~1 s contre 0,04 s en CSV | JSON tableau : à parser en entier avant de lire la première ligne |
| Analyser | Parquet | types conservés ; lecture 1,15 à 1,4 fois plus rapide que le CSV (écart faible, instable) | CSV : types à redéclarer à chaque lecture |
| Archiver | Parquet | 2,8 Mo contre 9,2 Mo, soit 3,3 fois plus léger, types inclus | CSV |

## 6. Questions (brouillon de Claude : à reformuler avec tes mots)

**(1) Quel type Pandas obtiens-tu pour `date_commande` selon le format, et pourquoi ?**
`str` en CSV, JSON et JSON Lines, `datetime64[us]` en Parquet. Les trois premiers sont du texte sans
type de date déclaré, donc Pandas lit des chaînes. Parquet enregistre le schéma dans le fichier et
le restitue ; il a gardé des dates parce que je les avais parsées avant l'export, il ne convertit pas seul.

**(2) À partir de quel volume la différence devient-elle gênante sur ma machine ?**
À 200 000 lignes, le JSON (~1 s contre 0,04 s en CSV, soit ~21 fois) reste supportable une fois. Cela
devient gênant quand on relit souvent le fichier ou quand il approche la RAM (un JSON en tableau se charge
en entier). Je n'ai pas mesuré de volume plus grand : je ne donne pas de seuil chiffré.